In [1]:
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import warnings
warnings.filterwarnings("ignore")
import yfinance as yf
import pandas as pd
    
it_stocks = ['INFY.NS', 'WIPRO.NS', 'TCS.NS', 'HCLTECH.NS', 'TECHM.NS']

# Download historical stock data for the selected IT companies without limiting the date range
data = yf.download(it_stocks)

# Flatten multi-level columns to make them easier to work with
data.columns = [f"{col[1]}_{col[0]}" for col in data.columns]

# Print the first few rows of the data to verify
print(data.tail())

[*********************100%***********************]  5 of 5 completed

                           HCLTECH.NS_Adj Close  INFY.NS_Adj Close  \
Date                                                                 
2024-12-06 00:00:00+00:00           1922.699951        1922.400024   
2024-12-09 00:00:00+00:00           1909.900024        1923.650024   
2024-12-10 00:00:00+00:00           1936.349976        1948.550049   
2024-12-11 00:00:00+00:00           1930.900024        1974.150024   
2024-12-12 00:00:00+00:00           1936.199951        1987.000000   

                           TCS.NS_Adj Close  TECHM.NS_Adj Close  \
Date                                                              
2024-12-06 00:00:00+00:00       4445.500000         1782.800049   
2024-12-09 00:00:00+00:00       4452.149902         1777.849976   
2024-12-10 00:00:00+00:00       4432.549805         1763.550049   
2024-12-11 00:00:00+00:00       4427.450195         1762.800049   
2024-12-12 00:00:00+00:00       4454.950195         1789.599976   

                           WIPRO.NS_Adj

In [ ]:
# Identify the first available data date for each stock
start_dates = {}
for ticker in it_stocks:
    adj_close_column = f"{ticker}_Adj Close"
    # Find the first non-null date for the stock
    first_valid_date = data[adj_close_column].dropna().index[0]
    start_dates[ticker] = first_valid_date

# Print out the start dates for each stock
print("\nStart dates for each stock:")
for ticker, start_date in start_dates.items():
    print(f"{ticker}: {start_date}")

# Now, filter the data to keep only the valid rows for each stock
valid_data = data.copy()

for ticker in it_stocks:
    # For each stock, filter the rows before its start date
    start_date = start_dates[ticker]
    adj_close_column = f"{ticker}_Adj Close"
    
    # Keep only rows from the stock's start date onward
    valid_data = valid_data.loc[valid_data.index >= start_date]
    
# Print the filtered data
print("\nFiltered Data:")
print(valid_data.tail())
print(valid_data.columns)

In [ ]:
# Print the missing counts before imputation
print("\nMissing values count (before imputation):")
missing_counts_before = valid_data.isnull().sum()
missing_counts_before = missing_counts_before[missing_counts_before > 0]  # Only show columns with missing values
for column, count in missing_counts_before.items():
    print(f"{column}: {count} missing values")

# Fill missing values using forward fill method (or you can choose other methods like bfill or interpolation)
valid_data_filled = valid_data.fillna(method='ffill')  # Forward fill

# Optionally, you can use other filling methods, for example:
# valid_data_filled = valid_data.fillna(method='bfill')  # Backward fill
# valid_data_filled = valid_data.interpolate()  # Linear interpolation

# Print the missing counts after imputation
print("\nMissing values count (after imputation):")
missing_counts_after = valid_data_filled.isnull().sum()
for column, count in missing_counts_after.items():
    print(f"{column}: {count} missing values")

In [ ]:
data_raw = valid_data_filled
data_raw.head()

In [ ]:
data_raw.info()

In [ ]:
data_raw[['HCLTECH.NS_Adj Close', 'INFY.NS_Adj Close']].plot(figsize=(10, 6))  # Plot some stocks' adjusted close prices

In [ ]:
def calculate_aps(row):
    total_volume = sum(row[f"{ticker}_Volume"] for ticker in it_stocks)
    
    aps = 0
    for ticker in it_stocks:
        weight = row[f"{ticker}_Volume"] / total_volume
        aps += row[f"{ticker}_Adj Close"] * weight
    
    return aps

# Apply the APS calculation to each row in the DataFrame
data_raw['APS'] = data_raw.apply(calculate_aps, axis=1)

# Save the updated DataFrame to CSV
data_raw.to_csv('IT_Stocks_Historical_Data_with_APS.csv', index=True)
print("Data with APS saved to 'IT_Stocks_Historical_Data_with_APS.csv'")

In [ ]:
# Define stock columns for scaling (those containing adjusted closing prices)
stock_columns = [col for col in valid_data_filled.columns if 'Adj Close' in col]

# Include APS column, which is already present in the dataframe
columns_to_scale = stock_columns + ['APS']

# Initialize the StandardScaler
scaler = StandardScaler()

# Apply scaling (standardization) to the selected columns (stock prices + APS)
valid_data_filled[columns_to_scale] = scaler.fit_transform(valid_data_filled[columns_to_scale])
data_scaled = valid_data_filled

# Check the data after scaling
print("Scaled data with APS included:")
print(data_scaled.head())

In [ ]:
# Save the scaled data with the APS column
data_scaled.to_csv('IT_Stocks_Historical_Data_Scaled_With_APS.csv', index=True)
print("Scaled data with APS saved to 'IT_Stocks_Historical_Data_Scaled_With_APS.csv'")

In [ ]:
data_monthly = data_raw.resample('M').mean()

# Plot the resampled data
plt.figure(figsize=(14, 8))

# Loop through the columns, plotting each one with reduced opacity for the others
for col in ['HCLTECH.NS_Adj Close', 'INFY.NS_Adj Close', 'TCS.NS_Adj Close', 'TECHM.NS_Adj Close', 'WIPRO.NS_Adj Close']:
    plt.plot(data_monthly.index, data_monthly[col], label=col, alpha=0.7, linewidth=0.8)  # Reduced opacity for other lines

# Plot the APS line with higher opacity and thicker line
plt.plot(data_monthly.index, data_monthly['APS'], label='APS', color='red', alpha=1.0, linewidth=1)  # APS with bold red line

# Set plot title and labels
plt.title('Adjusted Closing Prices and APS Over Time (Monthly)')
plt.xlabel('Date')
plt.ylabel('Price')

# Add legend
plt.legend()

# Add a grid for better visualization
plt.grid()

# Show the plot
plt.show()

In [ ]:
# List of columns to plot
columns_to_plot = ['HCLTECH.NS_Adj Close', 'INFY.NS_Adj Close', 'TCS.NS_Adj Close', 'TECHM.NS_Adj Close', 'WIPRO.NS_Adj Close', 'APS']

# Create subplots with one row and as many columns as there are stocks + APS
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 12))

# Flatten axes array for easier iteration
axes = axes.flatten()

# Plot each distribution in its own subplot
for i, col in enumerate(columns_to_plot):
    sns.histplot(data_raw[col], kde=True, bins=30, ax=axes[i], alpha=0.6)
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel('Price')
    axes[i].set_ylabel('Frequency')

# Adjust layout for better spacing
plt.tight_layout()

# Show the plot
plt.show()
